# Medicare+ — Prescription Handwriting OCR
## Notebook 3 of 3: Evaluation & Error Analysis

**Goal:** Load the fine-tuned TrOCR model and evaluate it rigorously on the held-out test set. Produce numbers and visualizations suitable for your project report.

**Prerequisite:** You ran `02_train_trocr.ipynb` and have a saved model at `/content/drive/MyDrive/medicare_plus_ocr/models/trocr-prescription`.

### Outline
1. Install dependencies
2. Load model + test split
3. Generate predictions on the entire test set
4. Compute CER (Character Error Rate) and WER (Word Error Rate)
5. Exact-match accuracy (drug name correctly recognized)
6. Top-K beam-search accuracy (correct answer in top-K predictions)
7. Confusion examples — most common error patterns
8. Side-by-side visualization (image + ground truth + prediction)
9. Compare against the baseline (un-fine-tuned) TrOCR for a 'before vs after' chart

## 1. Install dependencies

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'evaluate>=0.4' 'jiwer>=3' pandas pillow matplotlib

## 2. Mount Drive, load model & test split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, torch
PROJECT_ROOT = '/content/drive/MyDrive/medicare_plus_ocr'
MODEL_DIR = f'{PROJECT_ROOT}/models/trocr-prescription'
PREPARED_DIR = f'{PROJECT_ROOT}/data/prepared'
BASE_MODEL = 'microsoft/trocr-base-handwritten'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from datasets import load_from_disk

processor = TrOCRProcessor.from_pretrained(MODEL_DIR)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR).to(device).eval()

splits = load_from_disk(PREPARED_DIR)
test_split = splits['test']
print(f'Test samples: {len(test_split)}')

## 3. Generate predictions on the test set

In [ ]:
from PIL import Image as PILImage
import pandas as pd
from tqdm.auto import tqdm

BATCH_SIZE = 16
MAX_LEN = 32

def predict_batch(images, num_beams=4, num_return_sequences=1):
    pixel_values = processor(images, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=MAX_LEN,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            early_stopping=True,
        )
    texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    return texts

preds, refs = [], []
for i in tqdm(range(0, len(test_split), BATCH_SIZE)):
    rows = test_split[i:i + BATCH_SIZE]
    images = [img.convert('RGB') for img in rows['image']]
    labels = [str(l).strip().lower() for l in rows['label']]
    out = predict_batch(images, num_beams=4, num_return_sequences=1)
    preds.extend([t.strip().lower() for t in out])
    refs.extend(labels)

results_df = pd.DataFrame({'reference': refs, 'prediction': preds})
results_df.head(10)

## 4. CER and WER on test set

In [ ]:
import evaluate

cer_metric = evaluate.load('cer')
wer_metric = evaluate.load('wer')

cer = cer_metric.compute(predictions=preds, references=refs)
wer = wer_metric.compute(predictions=preds, references=refs)
print(f'Test CER: {cer:.4f}  (lower is better)')
print(f'Test WER: {wer:.4f}')

## 5. Exact-match accuracy

In [ ]:
exact = (results_df['reference'] == results_df['prediction']).mean()
print(f'Exact-match accuracy: {exact:.4f}  ({int(exact * len(results_df))} / {len(results_df)})')

## 6. Top-K accuracy via beam search

Generate top-5 candidates per image; the reference is considered 'found' if it appears anywhere in the top-K.

In [ ]:
K = 5
topk_hits = 0
topk_preds_all = []

for i in tqdm(range(0, len(test_split), BATCH_SIZE)):
    rows = test_split[i:i + BATCH_SIZE]
    images = [img.convert('RGB') for img in rows['image']]
    labels = [str(l).strip().lower() for l in rows['label']]
    out = predict_batch(images, num_beams=K, num_return_sequences=K)
    out = [[t.strip().lower() for t in out[j*K:(j+1)*K]] for j in range(len(images))]
    for label, candidates in zip(labels, out):
        if label in candidates:
            topk_hits += 1
        topk_preds_all.append(candidates)

print(f'Top-{K} accuracy: {topk_hits / len(test_split):.4f}')

## 7. Most common errors

In [ ]:
errors_df = results_df[results_df['reference'] != results_df['prediction']].copy()
errors_df['pair'] = errors_df['reference'] + '  →  ' + errors_df['prediction']
print(f'Total errors: {len(errors_df)} / {len(results_df)}')
print('\nTop 20 most frequent confusion pairs:')
print(errors_df['pair'].value_counts().head(20).to_string())

## 8. Side-by-side visualization

In [ ]:
import matplotlib.pyplot as plt
import random

N_SHOW = 12
sample_idx = random.sample(range(len(test_split)), N_SHOW)
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for ax, idx in zip(axes.ravel(), sample_idx):
    row = test_split[idx]
    img = row['image'].convert('RGB')
    label = str(row['label']).strip().lower()
    pred = preds[idx]
    color = 'green' if pred == label else 'red'
    ax.imshow(img)
    ax.set_title(f'true: {label}\npred: {pred}', fontsize=9, color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Before vs after — baseline TrOCR comparison

Load the un-fine-tuned base model and re-score on a sample of the test set. This is your 'fine-tuning lifted accuracy by X%' chart for your report.

In [ ]:
base_processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
base_model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL).to(device).eval()

def predict_with(p, m, images):
    pixel_values = p(images, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids = m.generate(pixel_values, max_length=MAX_LEN, num_beams=4, early_stopping=True)
    return [t.strip().lower() for t in p.batch_decode(ids, skip_special_tokens=True)]

SAMPLE_N = min(500, len(test_split))
sample_idx = random.sample(range(len(test_split)), SAMPLE_N)

base_preds, ft_preds, gold = [], [], []
for i in tqdm(range(0, SAMPLE_N, BATCH_SIZE)):
    chunk = sample_idx[i:i + BATCH_SIZE]
    images = [test_split[j]['image'].convert('RGB') for j in chunk]
    labels = [str(test_split[j]['label']).strip().lower() for j in chunk]
    base_preds.extend(predict_with(base_processor, base_model, images))
    ft_preds.extend(predict_with(processor, model, images))
    gold.extend(labels)

base_cer = cer_metric.compute(predictions=base_preds, references=gold)
ft_cer = cer_metric.compute(predictions=ft_preds, references=gold)
base_exact = sum(p == g for p, g in zip(base_preds, gold)) / len(gold)
ft_exact = sum(p == g for p, g in zip(ft_preds, gold)) / len(gold)

print(f'{"Metric":<20}{"Baseline":<15}{"Fine-tuned":<15}{"Delta":<10}')
print(f'{"-"*60}')
print(f'{"CER (lower=better)":<20}{base_cer:<15.4f}{ft_cer:<15.4f}{ft_cer - base_cer:+.4f}')
print(f'{"Exact match (acc)":<20}{base_exact:<15.4f}{ft_exact:<15.4f}{ft_exact - base_exact:+.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(['Baseline', 'Fine-tuned'], [base_cer, ft_cer])
ax[0].set_title('CER (lower is better)')
ax[0].set_ylabel('CER')

ax[1].bar(['Baseline', 'Fine-tuned'], [base_exact, ft_exact])
ax[1].set_title('Exact-match accuracy')
ax[1].set_ylabel('Accuracy')
ax[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 10. Save evaluation report

Persist the predictions table and summary numbers to Drive so you can cite them later.

In [ ]:
import json
REPORT_DIR = f'{PROJECT_ROOT}/reports'
os.makedirs(REPORT_DIR, exist_ok=True)

results_df.to_csv(f'{REPORT_DIR}/test_predictions.csv', index=False)
summary = {
    'test_cer': cer,
    'test_wer': wer,
    'test_exact_match': exact,
    f'top_{K}_accuracy': topk_hits / len(test_split),
    'baseline_sample_cer': base_cer,
    'finetuned_sample_cer': ft_cer,
    'baseline_sample_exact': base_exact,
    'finetuned_sample_exact': ft_exact,
    'test_samples': len(test_split),
    'comparison_sample_size': SAMPLE_N,
}
with open(f'{REPORT_DIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f'\nSaved to: {REPORT_DIR}')

## Done — use these numbers in your project report

Key things to highlight:
* The CER and exact-match accuracy on a held-out test set
* The 'before vs after fine-tuning' bar chart (notebook section 9)
* A handful of correct and incorrect prediction screenshots (notebook section 8)
* The most common confusion pairs (notebook section 7) — great for the 'limitations' section of your report